In [1]:
import cdflib
import pandas as pd
import numpy as np

In [7]:
def cdfconv_loconly(filename_a, sc_string):
    
    if sc_string=='C1':
        cdf_file_a = cdflib.CDF(filename_a)
        datetimes_a = cdflib.cdfepoch.encode(cdf_file_a['time_tags__C1_CP_FGM_FULL'])

        #convert strings to datetimes
        datetimes_series = pd.Series(datetimes_a) 
        datetimes_a_1 = pd.to_datetime(datetimes_series, errors = 'coerce')
        datetimes_a_1 = datetimes_a_1.to_frame(name="datetime")

        #check that file has data (otherwise causes error)
        shape_a = cdf_file_a['B_vec_xyz_gse__C1_CP_FGM_FULL'].shape

        #shape_a should be length 2. nest whole fnctn w/in this test.
        if len(shape_a) == 2:

            df_a = pd.DataFrame({'X_gse': cdf_file_a['sc_pos_xyz_gse__C1_CP_FGM_FULL'][:,0], 'Y_gse': cdf_file_a['sc_pos_xyz_gse__C1_CP_FGM_FULL'][:,1], 'Z_gse': cdf_file_a['sc_pos_xyz_gse__C1_CP_FGM_FULL'][:,2]})
            df_a = df_a.join(datetimes_a_1)

            df_a = df_a.set_index('datetime')
            
            df_a['R_GSE'] = (df_a['X_gse']**2 + df_a['Y_gse']**2 + df_a['Z_gse']**2)**0.5

            df_a['X_gse'] = df_a['X_gse']/6371
            df_a['Y_gse'] = df_a['Y_gse']/6371
            df_a['Z_gse'] = df_a['Z_gse']/6371
            df_a['R_GSE'] = df_a['R_GSE']/6371

        else:
            print(filename_a, 'data error')
            #return empty df
            df_a = pd.DataFrame({'A' : []})

        return df_a

In [8]:
#folder with input CDFs
#importing original Cluster CDFs to get GSE average values

path = '/Users/roseatkinson/Documents/CSA_Download_20260211_1852/C1_CP_FGM_FULL/C1_CP_FGM_FULL__20010423_000000_20010423_120000_V140306.cdf'

cluster_cdf = cdfconv_loconly(path, 'C1')


In [10]:
cluster_cdf

,X_gse,Y_gse,Z_gse,R_GSE
datetime,,,,
2001-04-23 00:00:00.035,12.005399,-15.353744,0.614535,19.499865
2001-04-23 00:00:00.079,12.005399,-15.353744,0.614519,19.499865
2001-04-23 00:00:00.124,12.005399,-15.353744,0.614519,19.499865
2001-04-23 00:00:00.169,12.005399,-15.353744,0.614503,19.499865
2001-04-23 00:00:00.213,12.005383,-15.353744,0.614503,19.499853
...,...,...,...,...
2001-04-23 11:59:59.819,7.099184,-13.185403,-5.660964,16.009365
2001-04-23 11:59:59.863,7.099184,-13.185388,-5.660964,16.009352
2001-04-23 11:59:59.908,7.099168,-13.185388,-5.660980,16.009352


In [11]:
#now match with relevant days/times from processed CSV, adding GSE columns to processed CSV!
#just for the example for now

int_csv_path = '/Users/roseatkinson/Documents/Testing/2001-04-22 22:40:00_C1 copy.csv'

test_df = pd.read_csv(int_csv_path ,encoding='utf-8')
test_df['datetime'] = pd.to_datetime(test_df['datetime'], format='mixed')
windows_list = test_df['datetime'].values.tolist()
test_df.set_index('datetime', inplace = True)

In [ ]:

def gse_ave(only_full_windows, raw_data_df):
    
    time_forward = dt.timedelta(minutes=4)

    cl_XGSE_ave = []
    cl_XGSE_med = []
    cl_YGSE_ave = []
    cl_YGSE_med = []
    cl_ZGSE_ave = []
    cl_ZGSE_med = []

    for i in only_full_windows:
    
        start_time = i
        end_time = i + time_forward
        
        mask = raw_data_df.loc[(raw_data_df.index >= start_time) & (raw_data_df.index < end_time)]
        
        mean_X_GSE = mask.loc[:,'X_gse'].mean()
        median_X_GSE = mask.loc[:,'X_gse'].median()

        mean_Y_GSE = mask.loc[:,'Y_gse'].mean()
        median_Y_GSE = mask.loc[:,'Y_gse'].median()

        mean_Z_GSE = mask.loc[:,'Z_gse'].mean()
        median_Z_GSE = mask.loc[:,'Z_gse'].median()

        cl_XGSE_ave.append(mean_X_GSE)
        cl_XGSE_med.append(median_X_GSE)
        cl_YGSE_ave.append(mean_Y_GSE)
        cl_YGSE_med.append(median_Y_GSE)
        cl_ZGSE_ave.append(mean_Z_GSE)
        cl_ZGSE_med.append(median_Z_GSE)
        
    gse_averages = pd.DataFrame({'datetime': only_full_windows, 'X GSE (mean)': cl_XGSE_ave, 'Y GSE (mean)': cl_YGSE_ave, 'Z GSE (mean)': cl_ZGSE_ave, })
    gse_averages['datetime'] = pd.to_datetime(gse_averages['datetime'])
    gse_averages = gse_averages.set_index('datetime')
        
    return(gse_averages)